# Does timing misalignment hurt a real robot policy? (ACT on ALOHA)

This notebook trains the **ACT** policy from Hugging Face LeRobot on the official ALOHA *transfer cube*
simulation benchmark three times, and measures task success in the simulator:

| condition | training data |
|---|---|
| `clean` | the original 50 demonstrations |
| `mixed` | episodes 25–49 have their actions logged **2 frames late** (what an older/different recording pipeline produces) |
| `repaired` | the `mixed` data after `demodyne realign` measured and undid the offset, **without** being told it |

**Runtime needed:** a GPU. In Colab: *Runtime → Change runtime type → T4 GPU*. On Kaggle: *Settings → Accelerator → GPU T4*.
**Time:** about 1 hour per condition with the defaults (3 conditions plus evaluation is about 3–3.5 hours). Kaggle
(*Save Version → Save & Run All*) keeps running when you close the tab; Colab free sessions may disconnect, but
the notebook resumes and skips finished conditions if `WORKDIR` is on Google Drive (see the next cell).

When it finishes, the last cell prints a results table. Please paste it into a GitHub issue at
https://github.com/shoggi211/demodyne/issues.

In [ ]:
# ---- settings -------------------------------------------------------------
STEPS = 20_000          # training steps per condition (ACT paper-level results need ~80k; 20k shows the trend)
BATCH = 8
EVAL_EPISODES = 50      # simulator episodes per condition
SEED = 1000
SHIFT_FRAMES = -2.0     # how late the second half's actions are logged
USE_GOOGLE_DRIVE = False  # Colab only: True keeps results across disconnects
WORKDIR = "/content/realign_act" if USE_GOOGLE_DRIVE is False else "/content/drive/MyDrive/realign_act"

In [ ]:
import os, sys, subprocess, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
if not IN_COLAB and WORKDIR.startswith("/content"):
    WORKDIR = os.path.abspath("realign_act")   # Kaggle / local
os.makedirs(WORKDIR, exist_ok=True)
os.environ["MUJOCO_GL"] = "egl"
print("working directory:", WORKDIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU: switch the runtime to a GPU first"

In [ ]:
# Install LeRobot, the ALOHA simulator and demodyne. Takes 3-5 minutes.
!pip -q install "lerobot[dataset,scipy-dep]==0.6.1" "git+https://github.com/shoggi211/demodyne"
# The ALOHA simulator needs dm_control, whose dependency `labmaze` (DeepMind maze levels, unused here)
# has no prebuilt package for recent Pythons and fails to compile, so install dm_control without it.
!pip -q install --no-deps "dm_control==1.0.41" "gym-aloha==0.1.5"
!pip -q install "mujoco==3.8.1" absl-py dm-env dm-tree glfw lxml pyopengl pyparsing imageio gymnasium
!apt-get -qq install -y libegl1 > /dev/null 2>&1 || true
import gym_aloha  # fails here, early, if the simulator is not usable
print("install OK")

## 1. Build the three datasets

In [ ]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import demodyne as dd
from demodyne.realign import realign, measure_alignment

SRC = f"{WORKDIR}/data/clean"
MIXED = f"{WORKDIR}/data/mixed"
REPAIRED = f"{WORKDIR}/data/repaired"
LATE = list(range(25, 50))

if not os.path.exists(f"{SRC}/meta/info.json"):
    LeRobotDataset("lerobot/aloha_sim_transfer_cube_human", root=SRC)          # downloads ~70 MB
if not os.path.exists(f"{MIXED}/meta/info.json"):
    realign(SRC, MIXED, shift_frames=SHIFT_FRAMES, episodes=LATE, copy_videos=True)

m = dd.load(MIXED)
ref = measure_alignment(m.subset(range(25)))
late = measure_alignment(m.subset(LATE))
print(f"measured alignment  first half {ref:+.2f} frames | second half {late:+.2f} frames")
if not os.path.exists(f"{REPAIRED}/meta/info.json"):
    rec = realign(MIXED, REPAIRED, target_frames=ref, episodes=LATE, copy_videos=True)
    print(f"demodyne applied a correction of {rec['shift_frames']:+.2f} frames (true answer {-SHIFT_FRAMES:+.2f})")
r = dd.load(REPAIRED)
print(f"after repair        first half {measure_alignment(r.subset(range(25))):+.2f} | second half {measure_alignment(r.subset(LATE)):+.2f}")
DATASETS = {"clean": SRC, "mixed": MIXED, "repaired": REPAIRED}

## 2. Train ACT on each (skips conditions that already finished)

In [ ]:
def checkpoint(cond):
    d = f"{WORKDIR}/runs/{cond}/checkpoints"
    if not os.path.isdir(d):
        return None
    steps = sorted(s for s in os.listdir(d) if s.isdigit())
    return f"{d}/{steps[-1]}/pretrained_model" if steps and int(steps[-1]) >= STEPS else None

for cond, root in DATASETS.items():
    if checkpoint(cond):
        print(cond, "already trained"); continue
    out = f"{WORKDIR}/runs/{cond}"
    ck = f"{out}/checkpoints"
    done = sorted(s for s in os.listdir(ck) if s.isdigit()) if os.path.isdir(ck) else []
    resume = bool(done)
    cmd = ["lerobot-train", f"--dataset.repo_id=local/aloha_{cond}", f"--dataset.root={root}",
           "--policy.type=act", "--policy.device=cuda", "--policy.push_to_hub=false",
           "--env.type=aloha", "--env.task=AlohaTransferCube-v0",
           f"--steps={STEPS}", f"--batch_size={BATCH}", f"--seed={SEED}", "--save_freq=5000",
           "--log_freq=500", "--num_workers=4", "--wandb.enable=false", f"--output_dir={out}"]
    if resume:
        cmd = ["lerobot-train", f"--config_path={ck}/{done[-1]}/pretrained_model/train_config.json",
               "--resume=true", f"--steps={STEPS}"]
    print("training", cond, "(resume)" if resume else ""); sys.stdout.flush()
    subprocess.run(cmd, check=True)

## 3. Evaluate each policy in the simulator

In [ ]:
results = {}
for cond in DATASETS:
    out = f"{WORKDIR}/eval/{cond}"
    info = f"{out}/eval_info.json"
    if not os.path.exists(info):
        subprocess.run(["lerobot-eval", f"--policy.path={checkpoint(cond)}", "--policy.device=cuda",
                        "--env.type=aloha", "--env.task=AlohaTransferCube-v0",
                        f"--eval.n_episodes={EVAL_EPISODES}", "--eval.batch_size=10", f"--seed={SEED}",
                        f"--output_dir={out}"], check=True)
    o = json.load(open(info))
    o = o.get("overall", o.get("aggregated", o))
    results[cond] = {"success_%": o["pc_success"], "avg_max_reward_of_4": o["avg_max_reward"]}
json.dump(results, open(f"{WORKDIR}/results.json", "w"), indent=1)

## 4. Results (please paste into a GitHub issue)

In [ ]:
import platform, torch
print(f"ACT, {STEPS} steps, batch {BATCH}, seed {SEED}, {EVAL_EPISODES} eval episodes, shift {SHIFT_FRAMES} frames on episodes 25-49")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} | demodyne {dd.__version__}")
print(f"{'condition':10s} {'success %':>10s} {'avg max reward (0-4)':>22s}")
for c, v in results.items():
    print(f"{c:10s} {v['success_%']:10.1f} {v['avg_max_reward_of_4']:22.2f}")
print()
print("Single seed: differences under ~10 points of success are within run-to-run noise.")